In [1]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.0 MB/s eta 0:00:00:00:0100:01


In [15]:
# ============================================================
# CELLULE 2 : Imports
# ============================================================
import pandas as pd
import numpy as np
import re
import warnings
import time
warnings.filterwarnings("ignore")
 
from sentence_transformers import SentenceTransformer
import faiss

In [3]:
# ============================================================
# CELLULE 3 : Chargement et préparation des données
# ============================================================
CSV_PATH = "/kaggle/input/datasets/medaymanelkajdouhi/code-route-maroc-nlp/export_final.csv"  # ← adapter
 
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
print(f"Dataset : {len(df)} articles, {len(df.columns)} colonnes")
 
# Créer un texte enrichi pour chaque article (métadonnées + contenu)
def build_chunk(row):
    """
    Combine les métadonnées structurées avec le texte brut.
    Le retriever cherchera dans ce texte enrichi.
    """
    parts = [f"المادة {row['article_id']}"]
    
    if pd.notna(row.get('chapitre')):
        parts.append(f"({row['chapitre']})")
    if pd.notna(row.get('type_paragraphe')):
        parts.append(f"[{row['type_paragraphe']}]")
    if pd.notna(row.get('amende_min')):
        if pd.notna(row.get('amende_max')) and row['amende_min'] != row['amende_max']:
            parts.append(f"غرامه: {int(row['amende_min'])} - {int(row['amende_max'])} درهم")
        else:
            parts.append(f"غرامه: {int(row['amende_min'])} درهم")
    if pd.notna(row.get('sanction_type')):
        parts.append(f"عقوبه: {row['sanction_type']}")
    if pd.notna(row.get('mots_cles')):
        parts.append(f"كلمات: {row['mots_cles']}")
    
    header = " | ".join(parts)
    body = str(row.get('texte_complet', ''))
    
    return f"{header}\n{body}"
 
df['chunk_text'] = df.apply(build_chunk, axis=1)

def clean_chunk_text(text):
    """Supprime le bruit des références législatives dans les chunks."""
    # Supprimer les références aux décrets modificatifs
    text = re.sub(r'من \d+ بتاريخ[\d.]+ الصادر بتنفيذه.*?ص\)', '', text)
    text = re.sub(r'\d+بتاريخ[\d.]+.*?ج\.ر عدد.*?ص\)', '', text)
    text = re.sub(r'ج\.ر عدد\s*\d+.*?ص\s*\d+', '', text)
    # Supprimer les numéros de page orphelins
    text = re.sub(r'^\d{1,3}\n', '', text, flags=re.MULTILINE)
    # Supprimer les lignes "الامانه العامه للحكومه"
    text = re.sub(r'الامانه العامه للحكومه', '', text)
    # Nettoyer espaces multiples
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()
 
# Chunking : articles longs (>1000 chars) sont découpés
MAX_CHUNK = 800
chunks = []
chunk_metadata = []
 
for _, row in df.iterrows():
    text = clean_chunk_text(row['chunk_text'])    # ← ajouté
    aid = row['article_id']
    
    if len(text) <= MAX_CHUNK:
        chunks.append(text)
        chunk_metadata.append({'article_id': aid, 'chunk_idx': 0})
    else:
        # Découpage par paragraphes
        paragraphs = text.split('\n')
        current = ""
        idx = 0
        for p in paragraphs:
            if len(current) + len(p) > MAX_CHUNK and current:
                chunks.append(current.strip())
                chunk_metadata.append({'article_id': aid, 'chunk_idx': idx})
                idx += 1
                current = p + "\n"
            else:
                current += p + "\n"
        if current.strip():
            chunks.append(current.strip())
            chunk_metadata.append({'article_id': aid, 'chunk_idx': idx})
 
print(f"Chunks créés : {len(chunks)} (depuis {len(df)} articles)")
print(f"Longueur moyenne : {np.mean([len(c) for c in chunks]):.0f} chars")


Dataset : 318 articles, 16 colonnes
Chunks créés : 462 (depuis 318 articles)
Longueur moyenne : 521 chars


In [4]:
EMBED_MODEL = "intfloat/multilingual-e5-base"
print(f"Chargement embedding model : {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)

print("Encodage des chunks...")
t0 = time.time()
chunks_for_encoding = [f"passage: {c}" for c in chunks]
embeddings = embed_model.encode(chunks_for_encoding, show_progress_bar=True, batch_size=32)
print(f"Encodage terminé en {time.time()-t0:.1f}s — shape: {embeddings.shape}")

Chargement embedding model : intfloat/multilingual-e5-base


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Encodage des chunks...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Encodage terminé en 5.2s — shape: (462, 768)


In [5]:

# ============================================================
# CELLULE 5 : Index FAISS
# ============================================================
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))
print(f"FAISS index : {index.ntotal} vecteurs, dimension {dimension}")
 

FAISS index : 462 vecteurs, dimension 768


In [6]:
# ============================================================
# CELLULE 6 : Retriever
# ============================================================
def retrieve(query, k=5):
    query_vec = embed_model.encode([f"query: {query}"])
    distances, indices = index.search(np.array(query_vec).astype('float32'), k)
    
    results = []
    for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        if idx < len(chunks):
            results.append({
                'rank': i + 1,
                'text': chunks[idx],
                'article_id': chunk_metadata[idx]['article_id'],
                'distance': float(dist),
                'score': 1 / (1 + float(dist)),
            })
    return results
 
# Test retriever
print("\n=== Test Retriever ===")
test_queries = [
    "ما هي عقوبه تجاوز السرعه؟",          # excès de vitesse
    "ما هي غرامه استعمال الهاتف؟",          # téléphone
    "هل حزام السلامه اجباري؟",             # ceinture
    "عقوبه السياقه في حاله سكر",           # alcool
]
 
for q in test_queries:
    results = retrieve(q, k=3)
    print(f"\nQ: {q}")
    for r in results:
        print(f"  [{r['rank']}] Art.{r['article_id']} (score={r['score']:.3f}) : {r['text'][:80]}...")



=== Test Retriever ===

Q: ما هي عقوبه تجاوز السرعه؟
  [1] Art.178 (score=0.724) : المادة 178 | (الكتاب الثاني / القسم الثاني / الباب الثاني) | [sanction] | عقوبه:...
  [2] Art.150 (score=0.719) : المادة 150 | (الكتاب الثاني / القسم الثاني / الباب الثاني) | [sanction] | عقوبه:...
  [3] Art.168 (score=0.716) : اعلاه للعقوبه 167يتعرض ايضا مرتكبو المخالفات المنصوص عليها في الفقره الثانيه من ...

Q: ما هي غرامه استعمال الهاتف؟
  [1] Art.185 (score=0.728) : المادة 185 | (الكتاب الثاني / القسم الثاني / الباب الثالث) | [sanction] | عقوبه:...
  [2] Art.164 (score=0.723) : ) درهم، كل مشغل لسايق مركبه 200.000) الي مايتي الف100.000يعاقب بغرامه من مايه ال...
  [3] Art.159 (score=0.719) : المادة 159 | (الكتاب الثاني / القسم الثاني / الباب الثاني) | [sanction] | عقوبه:...

Q: هل حزام السلامه اجباري؟
  [1] Art.99 (score=0.739) : كيلومترا في الساعه.30 ولا يتجاوز2
سياقه الدراجات الناريه او الدراجات ثلاثيه العج...
  [2] Art.93 (score=0.732) : المادة 93 | (القسم الثالث / الباب الثاني) | [obligation]
يجب

In [7]:
 
# ============================================================
# CELLULE 7 : Chargement LLM
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
 
def load_llm(model_name):
    """Charge un LLM et retourne un pipeline de génération."""
    print(f"Chargement {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        device_map="auto",
        torch_dtype="auto",
    )
    gen = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
    )
    print(f"  ✓ {model_name} chargé")
    return gen
 
# Charger 3 LLMs pour comparaison
# Sur Kaggle GPU T4 gratuit, ces modèles tiennent :
MODELS = {
    "Qwen-3B":    "Qwen/Qwen2.5-3B-Instruct",    # meilleur, mais vérifie la RAM GPU
}
 
generators = {}
for name, path in MODELS.items():
    try:
        generators[name] = load_llm(path)
    except Exception as e:
        print(f"  ✗ {name} : {e}")
 
print(f"\n{len(generators)} modèles chargés : {list(generators.keys())}")
 

Chargement Qwen/Qwen2.5-3B-Instruct...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  ✓ Qwen/Qwen2.5-3B-Instruct chargé

1 modèles chargés : ['Qwen-3B']


In [8]:

# ============================================================
# CELLULE 8 : Pipeline RAG
# ============================================================
def detect_hors_domaine(query):
    """
    Détection basique de questions hors domaine (code de la route).
    Retourne True si la question semble hors sujet.
    """
    route_keywords = [
        'سياقه', 'سرعه', 'مركبه', 'سياره', 'طريق', 'رخصه', 'غرامه',
        'سايق', 'وقوف', 'حزام', 'كحول', 'هاتف', 'دراجه', 'حادث',
        'شاحن', 'اشاره', 'تجاوز', 'ماده', 'عقوبه', 'نقط', 'حبس',
        'vitesse', 'amende', 'permis', 'route', 'conduire', 'véhicule',
        'infraction', 'sanction', 'code', 'circulation', 'stationnement',
        'ceinture', 'alcool', 'téléphone', 'feu', 'stop', 'priorité',
    ]
    query_lower = query.lower()
    return not any(kw in query_lower for kw in route_keywords)
 
def rag_answer(query, generator, k=5, max_tokens=300):
    """
    Pipeline RAG complet :
    1. Détection hors domaine
    2. Retrieval (FAISS)
    3. Construction du prompt
    4. Génération
    """
    # 1. Hors domaine ?
    if detect_hors_domaine(query):
        return {
            'answer': "هذا السؤال خارج نطاق مدونة السير على الطرق. يرجى طرح سؤال متعلق بقانون السير المغربي.",
            'hors_domaine': True,
            'sources': [],
            'query': query,
        }
    
    # 2. Retrieval
    results = retrieve(query, k=k)
    context = "\n\n".join([
        f"[المادة {r['article_id']}] {r['text']}" 
        for r in results
    ])
    sources = [r['article_id'] for r in results]
    
    # 3. Prompt
    prompt = f"""أنت مساعد قانوني متخصص في مدونة السير على الطرق المغربية (القانون رقم 52.05).
أجب على السؤال التالي بناءً على النصوص القانونية المقدمة فقط.
اذكر أرقام المواد المرجعية في إجابتك.
إذا لم تجد الإجابة في النصوص المقدمة، قل ذلك بوضوح.
 
النصوص القانونية:
{context}
 
السؤال: {query}
 
الإجابة:"""
    
    # 4. Génération
    output = generator(
        prompt,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.3,
        return_full_text=False,
    )
    
    answer_text = output[0]["generated_text"].strip()
    
    return {
        'answer': answer_text,
        'hors_domaine': False,
        'sources': sources,
        'query': query,
    }
 
# Test RAG
if generators:
    first_gen_name = list(generators.keys())[0]
    first_gen = generators[first_gen_name]
    
    print(f"=== Test RAG avec {first_gen_name} ===\n")
    
    test = rag_answer("ما هي عقوبه تجاوز السرعه؟", first_gen)
    print(f"Q: {test['query']}")
    print(f"Sources: Articles {test['sources']}")
    print(f"Réponse: {test['answer'][:300]}")
    
    # Test hors domaine
    print(f"\n--- Test hors domaine ---")
    test_hd = rag_answer("ما هو عدد سكان المغرب؟", first_gen)
    print(f"Q: {test_hd['query']}")
    print(f"Hors domaine: {test_hd['hors_domaine']}")
    print(f"Réponse: {test_hd['answer']}")
 

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Test RAG avec Qwen-3B ===

Q: ما هي عقوبه تجاوز السرعه؟
Sources: Articles [178, 150, 168, 154, 170]
Réponse: لا يمكن إيجاد الإجابة المباشرة حول عقوبة تجاوز السرعة في هذه النصوص القانونية المقدمة. 

المواد المرجعية المستخدمة: 
- المادة 178
- المادة 150
- المادة 168
- المادة 154
- المادة 170
الإجابة: لا يمكن إيجاد الإجابة المباشرة حول عقوبة تجاوز السرعة في هذه النصوص القانونية المقدمة. 

المواد المرجعية المس

--- Test hors domaine ---
Q: ما هو عدد سكان المغرب؟
Hors domaine: True
Réponse: هذا السؤال خارج نطاق مدونة السير على الطرق. يرجى طرح سؤال متعلق بقانون السير المغربي.


In [16]:

# ============================================================
# CELLULE 9 : Comparaison 3 LLMs
# ============================================================
eval_questions = [
    "ما هي عقوبه تجاوز السرعه القصوي؟",
    "ما هي غرامه الوقوف في مكان ممنوع؟",
    "هل يجب ارتداء حزام السلامه؟",
    "ما هي عقوبه السياقه تحت تاثير الكحول؟",
    "ما هي شروط الحصول علي رخصه السياقه؟",
    "ما هي عقوبه استعمال الهاتف اثناء السياقه؟",
    "ما هي انواع رخص السياقه؟",
    "ما هي عقوبه الفرار بعد حادثه سير؟",
]
 
if len(generators) >= 2:
    print("=== Comparaison des LLMs ===\n")
    
    comparison_results = []
    
    for q in eval_questions[:4]:  # 4 questions pour le temps
        print(f"\nQ: {q}")
        print("-" * 60)
        
        for name, gen in generators.items():
            t0 = time.time()
            result = rag_answer(q, gen, k=3, max_tokens=200)
            elapsed = time.time() - t0
            
            print(f"\n[{name}] ({elapsed:.1f}s)")
            print(f"  Sources: Art. {result['sources'][:3]}")
            print(f"  Réponse: {result['answer'][:150]}...")
            
            comparison_results.append({
                'question': q,
                'model': name,
                'answer': result['answer'],
                'sources': result['sources'],
                'time_s': elapsed,
            })
    
    comp_df = pd.DataFrame(comparison_results)
    print("\n\nTemps moyen par modèle :")
    print(comp_df.groupby('model')['time_s'].mean())
 

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [10]:

# ============================================================
# CELLULE 10 : Évaluation (Precision / Recall)
# ============================================================
# Ground truth : pour chaque question, quels articles DEVRAIENT être retournés ?
GROUND_TRUTH = {
    "ما هي عقوبه تجاوز السرعه القصوي؟": [175, 176, 185],
    "ما هي غرامه الوقوف في مكان ممنوع؟": [185, 186],
    "هل يجب ارتداء حزام السلامه؟": [177, 178],
    "ما هي عقوبه السياقه تحت تاثير الكحول؟": [167, 168, 169],
    "ما هي شروط الحصول علي رخصه السياقه؟": [10, 11, 14],
    "ما هي عقوبه استعمال الهاتف اثناء السياقه؟": [177, 178],
    "ما هي عقوبه الفرار بعد حادثه سير؟": [170, 171],
}
 
print("=== Évaluation Retriever (Precision@k, Recall@k) ===\n")
 
for k in [3, 5, 10]:
    precisions = []
    recalls = []
    
    for query, expected in GROUND_TRUTH.items():
        results = retrieve(query, k=k)
        retrieved_ids = [r['article_id'] for r in results]
        
        relevant_retrieved = len(set(retrieved_ids) & set(expected))
        precision = relevant_retrieved / k if k > 0 else 0
        recall = relevant_retrieved / len(expected) if expected else 0
        
        precisions.append(precision)
        recalls.append(recall)
    
    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)
    f1 = 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0
    
    print(f"  k={k:2d} : Precision={avg_p:.3f} | Recall={avg_r:.3f} | F1={f1:.3f}")
 
# Détail par question pour k=5
print(f"\nDétail pour k=5 :")
for query, expected in GROUND_TRUTH.items():
    results = retrieve(query, k=5)
    retrieved_ids = [r['article_id'] for r in results]
    hits = set(retrieved_ids) & set(expected)
    status = "✓" if hits else "✗"
    print(f"  {status} {query[:40]}...")
    print(f"    Attendu: {expected} | Récupéré: {retrieved_ids} | Hits: {list(hits)}")
 

=== Évaluation Retriever (Precision@k, Recall@k) ===

  k= 3 : Precision=0.048 | Recall=0.048 | F1=0.048
  k= 5 : Precision=0.114 | Recall=0.190 | F1=0.143
  k=10 : Precision=0.086 | Recall=0.310 | F1=0.134

Détail pour k=5 :
  ✓ ما هي عقوبه تجاوز السرعه القصوي؟...
    Attendu: [175, 176, 185] | Récupéré: [178, 150, 168, 184, 185] | Hits: [185]
  ✗ ما هي غرامه الوقوف في مكان ممنوع؟...
    Attendu: [185, 186] | Récupéré: [178, 184, 172, 166, 238] | Hits: []
  ✗ هل يجب ارتداء حزام السلامه؟...
    Attendu: [177, 178] | Récupéré: [93, 99, 48, 46, 185] | Hits: []
  ✓ ما هي عقوبه السياقه تحت تاثير الكحول؟...
    Attendu: [167, 168, 169] | Récupéré: [183, 167, 263, 169, 92] | Hits: [169, 167]
  ✓ ما هي شروط الحصول علي رخصه السياقه؟...
    Attendu: [10, 11, 14] | Récupéré: [240, 245, 9, 10, 132] | Hits: [10]
  ✗ ما هي عقوبه استعمال الهاتف اثناء السياقه...
    Attendu: [177, 178] | Récupéré: [263, 99, 167, 185, 216] | Hits: []
  ✗ ما هي عقوبه الفرار بعد حادثه سير؟...
    Attendu: [170, 171] | R

In [11]:

# ============================================================
# CELLULE 11 : Détection hors domaine (exemples)
# ============================================================
print("\n=== Test détection hors domaine ===\n")
hors_domaine_tests = [
    ("ما هي عقوبه تجاوز السرعه؟", False),         # ✓ dans le domaine
    ("ما هو عدد سكان المغرب؟", True),              # ✗ hors domaine
    ("كيف احصل علي رخصه السياقه؟", False),          # ✓ dans le domaine
    ("ما هي عاصمه فرنسا؟", True),                  # ✗ hors domaine
    ("ما هي غرامه القيادة بدون رخصة؟", False),     # ✓ dans le domaine
    ("كم سعر البنزين اليوم؟", True),               # ✗ hors domaine
]
 
correct = 0
for query, expected_hd in hors_domaine_tests:
    detected = detect_hors_domaine(query)
    status = "✓" if detected == expected_hd else "✗"
    if detected == expected_hd:
        correct += 1
    label = "hors domaine" if detected else "dans domaine"
    print(f"  {status} '{query[:35]}...' → {label}")
 
print(f"\nAccuracy détection : {correct}/{len(hors_domaine_tests)} ({100*correct/len(hors_domaine_tests):.0f}%)")
 


=== Test détection hors domaine ===

  ✓ 'ما هي عقوبه تجاوز السرعه؟...' → dans domaine
  ✓ 'ما هو عدد سكان المغرب؟...' → hors domaine
  ✓ 'كيف احصل علي رخصه السياقه؟...' → dans domaine
  ✓ 'ما هي عاصمه فرنسا؟...' → hors domaine
  ✓ 'ما هي غرامه القيادة بدون رخصة؟...' → dans domaine
  ✓ 'كم سعر البنزين اليوم؟...' → hors domaine

Accuracy détection : 6/6 (100%)


In [17]:
import gradio as gr

def gradio_rag(question):
    gen = list(generators.values())[0]
    result = rag_answer(question, gen, k=5, max_tokens=300)
    sources_text = ", ".join([f"المادة {s}" for s in result['sources']])
    if result['hors_domaine']:
        return result['answer'], " Question hors domaine"
    return result['answer'], f" Sources : {sources_text}"

demo = gr.Interface(
    fn=gradio_rag,
    inputs=gr.Textbox(
        label="سؤالك / Votre question",
        lines=3,
        placeholder="ما هي عقوبه تجاوز السرعه؟",
    ),
    outputs=[
        gr.Textbox(label="الإجابة / Réponse", lines=10),
        gr.Textbox(label="المراجع / Sources", lines=2),
    ],
    title=" مساعد مدونة السير المغربية",
    description="اسأل أي سؤال متعلق بمدونة السير المغربية (القانون 52.05) وسيجيبك المساعد بناءً على النصوص القانونية.",
    examples=[
        ["ما هي عقوبه تجاوز السرعه؟"],
        ["هل حزام السلامه اجباري؟"],
        ["ما هي عقوبه استعمال الهاتف اثناء السياقه؟"],
        ["ما هي عقوبه السياقه تحت تاثير الكحول؟"],
    ],
    theme=gr.themes.Soft(),
)
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://b95d4ac8d29bc2fd85.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [13]:
# Vérifier quels articles parlent de chaque sujet
topics = {
    "سرعه": "vitesse",
    "وقوف": "stationnement", 
    "حزام": "ceinture",
    "كحول": "alcool",
    "رخصه السياقه": "permis",
    "هاتف": "téléphone",
    "الفرار": "délit de fuite",
}

for keyword, label in topics.items():
    matching = df[df['texte_complet'].str.contains(keyword, na=False)]
    sanction_arts = matching[matching['type_paragraphe'] == 'sanction']['article_id'].tolist()
    print(f"{label:20s} ({keyword}) → sanctions: {sanction_arts[:8]}")

vitesse              (سرعه) → sanctions: [112, 164, 166, 167, 169, 172, 175, 176]
stationnement        (وقوف) → sanctions: [166, 167, 169, 172, 184, 185]
ceinture             (حزام) → sanctions: [184, 185]
alcool               (كحول) → sanctions: [166, 167, 169, 172, 183]
permis               (رخصه السياقه) → sanctions: [22, 28, 31, 35, 95, 98, 127, 129]
téléphone            (هاتف) → sanctions: [185]
délit de fuite       (الفرار) → sanctions: [182]
